# 03 – Bronze Data Profiling

**Purpose:** Explore all six existing Bronze tables before applying Silver transformations and data-quality rules.

This notebook is read-only. It does not modify, overwrite, or delete Bronze data.

**Approach:** Use simple PySpark DataFrame operations and direct imports. Profiling results are observations; they are not automatically Silver rejection rules.

## 1. Import libraries

In [0]:
from pyspark.sql.functions import (
    col, count, when, lit, countDistinct,
    min, max, avg, stddev, expr, to_timestamp
)

print("CARD TRANSACTION FRAUD MONITORING - BRONZE PROFILING")

## 2. Read the six existing Bronze tables

Use the existing catalog and schema. No CSV reload is needed.

In [0]:
transactions_df = spark.table("capstone_team01.bronze_schema.bronze_card_transactions")
status_changes_df = spark.table("capstone_team01.bronze_schema.bronze_payment_status_changes")
authentication_df = spark.table("capstone_team01.bronze_schema.bronze_device_authentication_events")

card_df = spark.table("capstone_team01.bronze_schema.bronze_card_master")
customer_df = spark.table("capstone_team01.bronze_schema.bronze_customer_master")
merchant_df = spark.table("capstone_team01.bronze_schema.bronze_merchant_master")

datasets = {
    "CARD TRANSACTIONS": transactions_df,
    "PAYMENT STATUS CHANGES": status_changes_df,
    "DEVICE AUTHENTICATION EVENTS": authentication_df,
    "CARD MASTER": card_df,
    "CUSTOMER MASTER": customer_df,
    "MERCHANT MASTER": merchant_df
}

print("All six Bronze tables loaded.")

## 3. Dataset overview

Review schema, sample records, and row counts for each dataset.

In [0]:
for name, df in datasets.items():
    print("\\n" + name)
    df.printSchema()
    print("Rows:", df.count())
    display(df.limit(5))

## 4. Business key profile

Expected business keys:
- Transactions: `transaction_id`
- Payment status events: `change_event_id`
- Authentication events: `auth_event_id`
- Master data: `card_id`, `customer_id`, `merchant_id`

Repeated `transaction_id` values in the event tables are allowed because one transaction can have multiple events.

In [0]:
key_columns = {
    "CARD TRANSACTIONS": (transactions_df, "transaction_id"),
    "PAYMENT STATUS CHANGES": (status_changes_df, "change_event_id"),
    "DEVICE AUTHENTICATION EVENTS": (authentication_df, "auth_event_id"),
    "CARD MASTER": (card_df, "card_id"),
    "CUSTOMER MASTER": (customer_df, "customer_id"),
    "MERCHANT MASTER": (merchant_df, "merchant_id")
}

key_rows = []

for name, (df, key) in key_columns.items():
    total_rows = df.count()
    null_ids = df.filter(col(key).isNull()).count()
    unique_ids = df.select(key).distinct().count()
    repeated_id_rows = total_rows - null_ids - unique_ids

    key_rows.append((name, key, total_rows, unique_ids, null_ids, repeated_id_rows))

key_summary = spark.createDataFrame(
    key_rows,
    "dataset STRING, key_column STRING, total_rows LONG, unique_ids LONG, null_ids LONG, repeated_id_rows LONG"
)

display(key_summary)

## 5. Exact duplicate row profile

Technical metadata columns such as `_source_file` and `_ingested_at` are excluded so the comparison focuses on source business columns.

In [0]:
duplicate_rows = []

for name, df in datasets.items():
    business_columns = [
        c for c in df.columns
        if not c.startswith("_") and c not in ["source_file", "ingestion_timestamp"]
    ]

    source_df = df.select(*business_columns)
    total_rows = source_df.count()
    distinct_rows = source_df.distinct().count()

    duplicate_rows.append((name, total_rows, distinct_rows, total_rows - distinct_rows))

duplicate_summary = spark.createDataFrame(
    duplicate_rows,
    "dataset STRING, total_rows LONG, distinct_rows LONG, duplicate_rows LONG"
)

display(duplicate_summary)

## 6. Null profile for every column

Shows null count and null percentage for every column in all six datasets.

In [0]:
null_rows = []

for name, df in datasets.items():
    total_rows = df.count()

    null_counts = df.select([
        count(when(col(c).isNull(), 1)).alias(c)
        for c in df.columns
    ]).first().asDict()

    for column_name, null_count in null_counts.items():
        null_pct = round(100.0 * null_count / total_rows, 2) if total_rows else 0.0
        null_rows.append((name, column_name, null_count, null_pct))

null_summary = spark.createDataFrame(
    null_rows,
    "dataset STRING, column_name STRING, null_count LONG, null_pct DOUBLE"
)

print("Columns containing nulls")
display(null_summary.filter(col("null_count") > 0).orderBy(col("null_count").desc()))

print("Complete null profile")
display(null_summary)

## 7. Categorical distributions

Review the observed values and counts. Do not treat an unfamiliar category as invalid until an allowed-value rule is confirmed.

In [0]:
category_columns = {
    "CARD TRANSACTIONS": (transactions_df, ["payment_status", "channel", "currency", "country"]),
    "PAYMENT STATUS CHANGES": (status_changes_df, ["old_status", "new_status", "reason_code", "source_system"]),
    "DEVICE AUTHENTICATION EVENTS": (authentication_df, ["authentication_method", "authentication_result", "ip_country", "risk_signal"]),
    "CARD MASTER": (card_df, ["card_type", "issuer_country", "card_status", "network", "tokenized_flag"]),
    "CUSTOMER MASTER": (customer_df, ["customer_segment", "home_country", "account_status", "kyc_status", "risk_band", "customer_type"]),
    "MERCHANT MASTER": (merchant_df, ["merchant_category", "merchant_country", "merchant_status", "settlement_currency", "risk_band", "acquirer"])
}

for name, (df, columns) in category_columns.items():
    print("\\n" + name)
    for column_name in columns:
        print("Distribution:", column_name)
        display(
            df.groupBy(column_name)
            .count()
            .orderBy(col("count").desc())
        )

## 8. Numeric profiling

Review ranges and summary statistics for numeric fields present in the source datasets.

In [0]:
print("TRANSACTION AMOUNT")
display(
    transactions_df.select(
        min("amount").alias("minimum_amount"),
        max("amount").alias("maximum_amount"),
        avg("amount").alias("average_amount"),
        expr("percentile_approx(amount, 0.5)").alias("median_amount"),
        stddev("amount").alias("standard_deviation")
    )
)

print("AUTHENTICATION ATTEMPT NUMBER")
display(
    authentication_df.select(
        min("attempt_number").alias("minimum_attempt"),
        max("attempt_number").alias("maximum_attempt"),
        avg("attempt_number").alias("average_attempt")
    )
)

print("PAYMENT CHANGE SEQUENCE")
display(
    status_changes_df.select(
        min("change_sequence").alias("minimum_sequence"),
        max("change_sequence").alias("maximum_sequence"),
        avg("change_sequence").alias("average_sequence")
    )
)

print("CARD DAILY LIMIT")
display(
    card_df.select(
        min("daily_limit").alias("minimum_daily_limit"),
        max("daily_limit").alias("maximum_daily_limit"),
        avg("daily_limit").alias("average_daily_limit")
    )
)

## 9. Date and timestamp ranges

Convert event timestamp strings for profiling. This does not change the Bronze tables.

In [0]:
timestamp_columns = [
    ("CARD TRANSACTIONS", transactions_df, "transaction_ts"),
    ("PAYMENT STATUS CHANGES", status_changes_df, "change_ts"),
    ("DEVICE AUTHENTICATION EVENTS", authentication_df, "event_ts")
]

for name, df, timestamp_column in timestamp_columns:
    print(name)
    display(
        df.select(
            min(to_timestamp(col(timestamp_column))).alias("earliest_timestamp"),
            max(to_timestamp(col(timestamp_column))).alias("latest_timestamp")
        )
    )

date_columns = [
    ("CARD MASTER", card_df, "issue_date"),
    ("CARD MASTER", card_df, "expiry_date"),
    ("CUSTOMER MASTER", customer_df, "onboarded_date"),
    ("MERCHANT MASTER", merchant_df, "onboarded_date")
]

for name, df, date_column in date_columns:
    print(name, date_column)
    display(
        df.select(
            min(col(date_column)).alias("earliest_date"),
            max(col(date_column)).alias("latest_date")
        )
    )

In [0]:
print("======================================")
print("CARD TRANSACTION FRAUD MONITORING")
print("BRONZE DATA PROFILING COMPLETED")
print("======================================")

for name, df in datasets.items():
    print(name, ":", df.count(), "rows")

In [0]:

from pyspark.sql.functions import col, count, when, lit, round as spark_round

# ---------- row counts ----------
summary_rows = []
for name, df in datasets.items():
    total = df.count()
    business_cols = [c for c in df.columns if not c.startswith("_") and c not in ["source_file", "ingestion_timestamp"]]
    distinct = df.select(*business_cols).distinct().count()
    dup = total - distinct
    null_cols = sum(1 for c in df.columns if df.filter(col(c).isNull()).count() > 0)
    summary_rows.append((name, total, distinct, dup, len(df.columns), null_cols))

summary_df = spark.createDataFrame(
    summary_rows,
    "dataset STRING, total_rows LONG, distinct_rows LONG, duplicate_rows LONG, total_columns INT, columns_with_nulls INT"
)

print("="*60)
print("  BRONZE PROFILING SUMMARY")
print("="*60)
print()
print("Table-level overview:")
display(summary_df)